In [ ]:
# training process
#!python train.py -data_dir dataset/train// -ENCODER resnet101 -model_dir weights1/  -augment -rect

In [1]:
import sys
import os
import torch
import torchvision.transforms as transforms
import cv2
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
import time
import numpy as np
import segmentation_models_pytorch as smp
import math

/home/xuechunliang/anaconda3/envs/xuechunliang/lib/python3.9/site-packages/requests/__init__.py:109: RequestsDependencyWarning: urllib3 (2.13.900) or chardet (5.2.0)/charset_normalizer (3.3.2) doesn't match a supported version!
  warnings.warn(


In [2]:
# define functions
def select_device(device='', batch_size=None):
    # device = 'cpu' or '0' or '0,1,2,3'
    cpu_request = device.lower() == 'cpu'
    if device and not cpu_request:  # if device requested other than 'cpu'
        os.environ['CUDA_VISIBLE_DEVICES'] = device  # set environment variable
        assert torch.cuda.is_available(), 'CUDA unavailable, invalid device %s requested' % device  # check availablity
    cuda = False if cpu_request else torch.cuda.is_available()
    if cuda:
        c = 1024 ** 2  # bytes to MB
        ng = torch.cuda.device_count()
        if ng > 1 and batch_size:  # check that batch_size is compatible with device_count
            assert batch_size % ng == 0, 'batch-size %g not multiple of GPU count %g' % (batch_size, ng)
        x = [torch.cuda.get_device_properties(i) for i in range(ng)]
        s = 'Using CUDA '  # apex for mixed precision https://github.com/NVIDIA/apex
        for i in range(0, ng):
            if i == 1:
                s = ' ' * len(s)
            print("%sdevice%g _CudaDeviceProperties(name='%s', total_memory=%dMB)" %
                  (s, i, x[i].name, x[i].total_memory / c))
    else:
        print('Using CPU')

    print('')  # skip a line
    return torch.device('cuda:0' if cuda else 'cpu')
def letterbox_unet(img, new_shape=(448, 448), color=(0,0, 0), auto=False, scaleFill=False, scaleup=False):
    shape = img.shape[:2]  # current shape [height, width]
#     print(shape)
    if isinstance(new_shape, int):
        new_shape = (new_shape, new_shape)

    # Scale ratio (new / old)
    r = min(new_shape[0] / shape[0], new_shape[1] / shape[1])
    if not scaleup:  # only scale down, do not scale up (for better test mAP)
        r = min(r, 1.0)

    # Compute padding
    ratio = r, r  # width, height ratios
    new_unpad = int(round(shape[1] * r)), int(round(shape[0] * r))
    dw, dh = new_shape[1] - new_unpad[0], new_shape[0] - new_unpad[1]  # wh padding
    if auto:  # minimum rectangle
        dw, dh = np.mod(dw, 64), np.mod(dh, 64)  # wh padding
    elif scaleFill:  # stretch
        dw, dh = 0.0, 0.0
        new_unpad = (new_shape[1], new_shape[0])
        ratio = new_shape[1] / shape[1], new_shape[0] / shape[0]  # width, height ratios
    dw /= 2  # divide padding into 2 sides
    dh /= 2

    if shape[::-1] != new_unpad:  # resize
        img = cv2.resize(img, new_unpad, interpolation=cv2.INTER_LINEAR)
    top, bottom = int(round(dh - 0.1)), int(round(dh + 0.1))
    left, right = int(round(dw - 0.1)), int(round(dw + 0.1))
    img = cv2.copyMakeBorder(img, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)  # add border
    return img, ratio, (dw, dh)
def evaluate_img(model, img, device, train_tfms, input_size):
    input_width, input_height = input_size[0], input_size[1]
    img_width, img_height = img.shape[:2]
    if input_width != img_width and input_height != img_height:
        img_1 = cv2.resize(img, (input_width, input_height), cv2.INTER_AREA)
    else:
        img_1 = img.copy()
    X = train_tfms(Image.fromarray(img_1))
    X = X.unsqueeze(0).to(device)  # [N, 1, H, W]

    mask = model(X)
    mask = torch.sigmoid(mask[0, 0]).data.cpu().numpy()
    if input_width != img_width and input_height != img_height:
        mask = cv2.resize(mask, (img_width, img_height), cv2.INTER_AREA)
    return mask
def evaluate_img_patch(model, img, device, train_tfms, input_size):
    input_width, input_height = input_size[0], input_size[1]
    img_height, img_width, img_channels = img.shape
    if img_width <= input_width and img_height <= input_height:
        #print('Image size equal to or less than 448 pixel:', (img_width, img_height))
        img_border, _, (dw, dh) = letterbox_unet(img)
        prob_map = evaluate_img(model, img_border, device, train_tfms, input_size)
        prob_map = prob_map[round(dh-0.1):round(448-round(dh+0.1)), round(dw-0.1):round(448-round(dw+0.1))]
        return prob_map
    else:
    
        x_num = math.ceil(img_width/input_width)
        y_num = math.ceil(img_height/input_height)
        normalization_map = np.zeros((img_height, img_width))
        for y in range(0, y_num):
            for x in range(0, x_num):
                x_coor = x*input_width
                y_coor = y*input_height
                if x_coor + input_width > img_width:
                    x_coor1 = img_width
                else:
                    x_coor1 = x_coor + input_width
                if y_coor + input_height > img_height:
                    y_coor1 = img_height
                else:
                    y_coor1 = y_coor + input_height
                if y == y_num-1 and y > 0:
                    y_coor = y_coor1 - input_height
                if x == x_num -1 and x > 0:
                    x_coor = x_coor1- input_width
                segment = img[y_coor:y_coor1, x_coor:x_coor1]
                if y_coor1-y_coor!= input_height or x_coor1-x_coor != input_width:
                    segment, _, (dw, dh) = letterbox_unet(segment)
                patch_n = train_tfms(Image.fromarray(segment))
                X = patch_n.unsqueeze(0).to(device)
                masks_pred = model(X)
                mask = torch.sigmoid(masks_pred[0, 0]).data.cpu().numpy()
                if y_coor1-y_coor!= input_height or x_coor1-x_coor != input_width:
                    mask = mask[round(dh-0.1):round(input_height-round(dh+0.1)), round(dw-0.1):round(input_width-round(dw+0.1))]
                normalization_map[y_coor:y_coor1, x_coor:x_coor1] = mask
        return normalization_map
def visualize(**images):
    """PLot images in one row."""
    n = len(images)
    plt.figure(figsize=(20, 10))
    for i, (name, image) in enumerate(images.items()):
        plt.subplot(1, n, i + 1)
        plt.xticks([])
        plt.yticks([])
        plt.title(' '.join(name.split('_')).title())
        plt.imshow(image)
    plt.show()

In [3]:
# Define device for GPU or CPU using, define model and some hyperparameters for inference.
device = select_device(device='1') # If use GPU, device = select_device(device = '0') or select the second GPU by device = select_device(device = '1')--->if the computer have 2 GPU
# The model is efficientnet backbone with Unet architecture.
model = smp.Unet(
    encoder_name='resnet101', 
    encoder_weights='imagenet', 
    classes=1, 
    activation=None,
    
    
)
# Load the trained model.
checkpoint = torch.load('weights/model_epoch_59.pt', map_location=device,weights_only=False)
model.load_state_dict(checkpoint['model'])
model.to(device)
model.eval()
# Preprocessing
channel_means = [0.485, 0.456, 0.406]
channel_stds  = [0.229, 0.224, 0.225]
train_tfms = transforms.Compose([transforms.ToTensor(), transforms.Normalize(channel_means, channel_stds)])
input_size = [448, 448]

Using CUDA device0 _CudaDeviceProperties(name='NVIDIA RTX 6000 Ada Generation', total_memory=48640MB)



In [4]:
#crack-width reconstruction
from skimage import morphology
from skimage.morphology import skeletonize
import numpy as np
import sknw
import networkx as nx

def recover_crack_width(prob_map_binary, crack_centerline, image_for_watershed):
    """
    Recover crack width using watershed guided by crack centerline.

    Args:
        prob_map_binary: binary crack mask (0/1)
        crack_centerline: pruned skeleton used as watershed seeds
        image_for_watershed: original RGB image used for watershed gradient

    Returns:
        refined_mask: width-recovered crack mask
    """

    # Convert binary mask to 0–255 image and apply threshold
    binary_img = prob_map_binary * 255
    watershed_img = image_for_watershed.copy()
    binary_img[binary_img < 10] = 0
    binary_img[binary_img >= 10] = 255

    # Extract connected components
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(binary_img, connectivity=8)
    labels = labels.astype(np.int8)

    region_masks = []

    for label_id in range(1, num_labels + 1):
        labels_copy = labels.copy()

        # Isolate one connected component
        component_mask = np.zeros((labels_copy.shape[0], labels_copy.shape[1]), dtype=np.uint8)
        component_mask[labels_copy == label_id] = 255

        # Extract corresponding centerline region
        centerline_region = np.zeros_like(crack_centerline, dtype=crack_centerline.dtype)
        centerline_region[component_mask == 255] = crack_centerline[component_mask == 255]

        sure_background = component_mask
        sure_foreground = centerline_region

        # Unknown region = background - foreground
        unknown_region = cv2.subtract(sure_background, sure_foreground)

        # Create watershed markers from centerline seeds
        _, markers = cv2.connectedComponents(sure_foreground)
        markers = markers + 1            # background = 1
        markers[unknown_region == 255] = 0

        # Apply watershed
        markers = cv2.watershed(watershed_img, markers)
        markers[markers > 1] = 255
        markers[markers <= 1] = 0

        region_masks.append(markers)

    # Merge all component results
    refined_mask = np.zeros((labels_copy.shape[0], labels_copy.shape[1]), dtype=np.uint8)
    for m in region_masks:
        refined_mask = cv2.add(refined_mask, m.astype(np.uint8))

    # Remove small regions
    refined_mask = morphology.remove_small_objects(refined_mask.astype(bool), min_size=50)

    # Close tiny gaps (<2 pixels)
    struct_elem = morphology.disk(1)
    refined_mask = morphology.closing(refined_mask, struct_elem)

    return refined_mask

def remove_short_branches_by_component(skel, length_threshold=10):
    """
    For a binary skeleton image (0/1), process each connected component:
    keep the main trunk (longest path) and prune branches shorter than threshold.

    Args:
        skel: binary skeleton image (0/1)
        length_threshold: branch length threshold in pixels

    Returns:
        pruned_skel: skeleton with short side branches removed
    """
    # Convert skeleton to graph
    graph = sknw.build_sknw(skel, multi=False)
    pruned_skel = skel.copy()

    # Iterate over connected components
    for comp_nodes in nx.connected_components(graph):
        subG = graph.subgraph(comp_nodes)
        if subG.number_of_edges() == 0:
            continue

        # 1. Compute longest path (graph diameter)
        arbitrary_node = list(subG.nodes())[0]
        distances = nx.single_source_dijkstra_path_length(subG, arbitrary_node, weight='weight')
        farthest_node = max(distances, key=distances.get)
        
        distances2 = nx.single_source_dijkstra_path_length(subG, farthest_node, weight='weight')
        other_end = max(distances2, key=distances2.get)
        
        try:
            longest_path = nx.dijkstra_path(subG, farthest_node, other_end, weight='weight')
        except Exception:
            continue
        
        longest_path_edges = set()
        for i in range(len(longest_path) - 1):
            longest_path_edges.add(frozenset([longest_path[i], longest_path[i+1]]))
        
        # 2. Remove short branches not on the main trunk
        for (s, e) in subG.edges():
            edge_data = subG[s][e]
            branch_length = edge_data.get('weight', 0)

            if frozenset([s, e]) in longest_path_edges:
                continue

            if branch_length < length_threshold:
                pts = np.array(edge_data['pts'], dtype=np.int32)
                pruned_skel[pts[:, 0], pts[:, 1]] = 0

    return pruned_skel


In [5]:
#This part of the code is used to refine the model predictions, including crack-width reconstruction and related post-processing

# Input / output directories
path_img_input  = 'images'   # original images
path_mask_output = 'masks'   # predicted masks

from skimage import morphology
from skimage.morphology import skeletonize
from scipy.ndimage import distance_transform_edt, maximum_filter

image_names = sorted([f for f in os.listdir(path_img_input) if f.endswith('.jpg')])

for image_name in image_names:
    # ---------------------------------------------------
    # 1. Read image
    # ---------------------------------------------------
    img_original = cv2.imread(path_img_input + image_name)
    img_for_model = np.asarray(img_original)
    img_for_watershed = np.asarray(img_original)

    # Resize large images for model inference
    if img_for_model.shape[0] > 448:
        resized_width = int(img_for_model.shape[1] * 448 * 2.0 / img_for_model.shape[0])
        resize_dim = (resized_width, int(448 * 2.0))
        img_for_model = cv2.resize(img_for_model, resize_dim, interpolation=cv2.INTER_AREA)

    # ---------------------------------------------------
    # 2. Model inference
    # ---------------------------------------------------
    prob_map_full = evaluate_img_patch(model, img_for_model, device, train_tfms, input_size)

    # Normalize probability map
    prob_map = prob_map_full / prob_map_full.max()

    # Threshold to binary mask
    prob_map[prob_map < 0.207] = 0
    prob_map[prob_map >= 0.207] = 1

    # Slight dilation to connect fragmented regions
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (3, 3))
    prob_map = cv2.dilate(prob_map, kernel, iterations=1)

    prob_map = prob_map.astype(np.uint8)

    # Resize back to original resolution
    prob_map = cv2.resize(prob_map, (img_for_watershed.shape[1], img_for_watershed.shape[0]))

    # ---------------------------------------------------
    # 3. Skeletonization + branch pruning
    # ---------------------------------------------------
    skeleton = morphology.skeletonize(prob_map / 255).astype(np.uint8)
    pruned_skeleton = remove_short_branches_by_component(skeleton, length_threshold=15)

    # ---------------------------------------------------
    # 4. Crack width reconstruction via watershed
    # ---------------------------------------------------
    prob_map = recover_crack_width(prob_map, pruned_skeleton, img_for_watershed)

    # ---------------------------------------------------
    # 5. Remove unrealistically wide cracks
    # ---------------------------------------------------
    binary_mask = prob_map > 0.5

    # Distance transform (distance to nearest background)
    distance_map = distance_transform_edt(binary_mask)

    # Estimate local crack width
    local_max_distance = maximum_filter(distance_map, size=20)
    local_width_map = local_max_distance * 2  # distance → width

    # Remove regions wider than 20 pixels
    mask_remove_wide = (local_width_map > 20) & binary_mask
    final_mask = binary_mask & (~mask_remove_wide)

    # Remove very small connected components
    final_mask = morphology.remove_small_objects(final_mask.astype(bool), min_size=300)

    # ---------------------------------------------------
    # 6. Save result
    # ---------------------------------------------------
    cv2.imwrite(path_mask_output + image_name, final_mask * 255)



KeyboardInterrupt

